# Prepare a verified official NOAA Storm Events cache

Run this notebook in Google Colab or locally. It downloads the unchanged 2019--2024 annual detail archives directly from NOAA, using anonymous NOAA FTP when the NCEI HTTPS host is unreachable. It validates their structure, records SHA-256 hashes and official URLs, and creates a ZIP for a private Kaggle Dataset. Do not use a third-party combined Storm Events file for the final study.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from ftplib import FTP
import csv
import gzip
import hashlib
import json
import re
import shutil

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

START_YEAR = 2019
END_YEAR = 2024
NOAA_INDEX = 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/'
NOAA_FTP_HOST = 'ftp.ncei.noaa.gov'
NOAA_FTP_PATH = '/pub/data/swdi/stormevents/csvfiles'
RUN_STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
CACHE_DIR = Path(f'noaa_storm_events_official_{START_YEAR}_{END_YEAR}_{RUN_STAMP}')
CACHE_DIR.mkdir(parents=True, exist_ok=False)

retry = Retry(
    total=1, connect=1, read=1, backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({'GET'}),
)
session = requests.Session()
session.headers['User-Agent'] = 'Event-TimeRAF/0.1 (academic cache preparation)'
session.mount('https://', HTTPAdapter(max_retries=retry))


In [ ]:
ftp = None
try:
    index_response = session.get(NOAA_INDEX, timeout=(10, 30))
    index_response.raise_for_status()
    index_names = set(re.findall(r'StormEvents_details-ftp_v1\.0_d\d{4}_c\d{8}\.csv\.gz', index_response.text))
    retrieval_transport = 'ncei_https'
except requests.RequestException as error:
    print(f'NCEI HTTPS unavailable ({error}); switching to official anonymous NOAA FTP.')
    ftp = FTP(NOAA_FTP_HOST, timeout=60)
    ftp.login()
    ftp.cwd(NOAA_FTP_PATH)
    index_names = set(ftp.nlst())
    retrieval_transport = 'ncei_anonymous_ftp'

def download_from_ftp(connection, name, destination):
    digest = hashlib.sha256()
    byte_count = 0
    with destination.open('wb') as handle:
        def receive(chunk):
            nonlocal byte_count
            handle.write(chunk)
            digest.update(chunk)
            byte_count += len(chunk)
        connection.retrbinary(f'RETR {name}', receive, blocksize=1024 * 1024)
    return byte_count, digest.hexdigest()

def gzip_payload_metadata(path):
    digest = hashlib.sha256()
    byte_count = 0
    with gzip.open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
            byte_count += len(chunk)
    return byte_count, digest.hexdigest()

records = []
required_columns = {
    'EVENT_ID', 'YEAR', 'STATE', 'CZ_FIPS', 'CZ_TYPE', 'CZ_NAME',
    'BEGIN_DATE_TIME', 'END_DATE_TIME', 'EVENT_TYPE',
}

for year in range(START_YEAR, END_YEAR + 1):
    pattern = re.compile(rf'StormEvents_details-ftp_v1\.0_d{year}_c\d{{8}}\.csv\.gz')
    names = sorted(name for name in index_names if pattern.fullmatch(name))
    if not names:
        raise RuntimeError(f'Official NOAA index has no detail archive for {year}')
    name = names[-1]
    url = NOAA_INDEX + name
    destination = CACHE_DIR / name
    if ftp is not None:
        byte_count, sha256 = download_from_ftp(ftp, name, destination)
        delivery_url = f'ftp://{NOAA_FTP_HOST}{NOAA_FTP_PATH}/{name}'
    else:
        digest = hashlib.sha256()
        byte_count = 0
        with session.get(url, stream=True, timeout=(30, 300)) as response:
            response.raise_for_status()
            with destination.open('wb') as handle:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        handle.write(chunk)
                        digest.update(chunk)
                        byte_count += len(chunk)
        sha256 = digest.hexdigest()
        delivery_url = url
    with gzip.open(destination, 'rt', encoding='utf-8-sig', newline='') as handle:
        columns = set(next(csv.reader(handle)))
    missing = required_columns - columns
    if missing:
        raise RuntimeError(f'{name} is missing expected columns: {sorted(missing)}')
    uncompressed_bytes, uncompressed_sha256 = gzip_payload_metadata(destination)
    records.append({
        'archive_year': year, 'name': name, 'url': url,
        'bytes': byte_count, 'sha256': sha256, 'delivery_url': delivery_url,
        'uncompressed_name': name[:-3],
        'uncompressed_bytes': uncompressed_bytes,
        'uncompressed_sha256': uncompressed_sha256,
    })
    print(f'{year}: {name} ({byte_count / 1024**2:.1f} MiB)')

if ftp is not None:
    ftp.quit()

manifest = {
    'schema_version': 1,
    'source_name': 'NOAA NCEI Storm Events',
    'official_index_url': NOAA_INDEX,
    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
    'retrieval_transport': retrieval_transport,
    'study_years': [START_YEAR, END_YEAR],
    'files': records,
}
manifest_path = CACHE_DIR / 'source_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
manifest


In [ ]:
zip_path = Path(shutil.make_archive(
    CACHE_DIR.name, 'zip', root_dir=CACHE_DIR.parent, base_dir=CACHE_DIR.name
))
print(f'Created {zip_path.resolve()} ({zip_path.stat().st_size / 1024**2:.1f} MiB)')

try:
    from google.colab import files
except ImportError:
    print('Download the ZIP shown above and upload it as a private Kaggle Dataset.')
else:
    files.download(str(zip_path))
